##### 06 - Analytics Queries on Gold Layer
This notebook demonstrates the business value of the lakehouse by running
analytical queries on the Gold layer tables. These queries represent
the kind of insights a data analyst or BI team would consume.

##### 1. Monthly Revenue Trend

In [0]:
%sql
SELECT
  d.month_name,
  d.month,
  COUNT(DISTINCT f.order_id) AS total_orders,
  COUNT(DISTINCT f.customer_id) AS unique_customers,
  ROUND(SUM(f.line_total), 2) AS gross_revenue,
  ROUND(SUM(CASE WHEN f.status = 'cancelled' THEN f.line_total ELSE 0 END), 2) AS cancelled_revenue,
  ROUND(SUM(f.line_total) - SUM(CASE WHEN f.status = 'cancelled' THEN f.line_total ELSE 0 END), 2) AS net_revenue,
  ROUND(AVG(f.line_total), 2) AS avg_line_value
FROM ecommerce_gold.fact_orders f
JOIN ecommerce_gold.dim_date d ON f.order_date_key = d.date_key
GROUP BY d.month_name, d.month
ORDER BY d.month


month_name,month,total_orders,unique_customers,gross_revenue,cancelled_revenue,net_revenue,avg_line_value
January,1,4280,3469,8.118067499E7,1.023952659E7,7.09411484E7,8932.73
February,2,3933,3236,7.138418312E7,6765118.88,6.461906424E7,8781.42
March,3,4131,3422,7.825087218E7,8620739.02,6.963013316E7,8949.09
April,4,4140,3455,7.574905042E7,1.022952522E7,6.55195252E7,8730.87
May,5,4299,3486,8.01646536E7,9062920.63,7.110173297E7,8865.81
June,6,4033,3335,7.568792582E7,7943910.34,6.774401548E7,8836.89
July,7,4387,3558,8.178549264E7,9404734.22,7.238075842E7,8919.78
August,8,4239,3454,7.905138716E7,8126219.37,7.092516779E7,8845.41
September,9,4028,3309,7.134338771E7,7907858.06,6.343552965E7,8472.08
October,10,4169,3395,7.776408008E7,8853511.56,6.891056852E7,8864.02


##### 2. Revenue by Category & Price Tier

In [0]:
%sql
SELECT
  p.category,
  p.price_tier,
  COUNT(DISTINCT f.order_id) AS orders,
  SUM(f.quantity) AS units_sold,
  ROUND(SUM(f.line_total), 2) AS revenue,
  ROUND(AVG(f.discount_pct) * 100, 1) AS avg_discount_pct
FROM ecommerce_gold.fact_orders f
JOIN ecommerce_gold.dim_products p ON f.product_id = p.product_id
WHERE f.status != 'cancelled'
GROUP BY p.category, p.price_tier
ORDER BY revenue DESC


category,price_tier,orders,units_sold,revenue,avg_discount_pct
Electronics,Luxury,8823,14436,5.9542891716E8,7.2
Sports,Premium,6488,10268,5.337604239E7,7.1
Home & Kitchen,Premium,6928,11082,4.671196182E7,7.1
Clothing,Premium,6158,9625,3.542831724E7,7.2
Home & Kitchen,Luxury,922,1410,1.905870404E7,7.2
Toys,Premium,3157,4917,1.253451251E7,7.1
Electronics,Premium,1791,2729,1.012904438E7,7.3
Clothing,Mid-Range,4933,7720,9382687.57,7.2
Toys,Mid-Range,5769,9172,8069331.22,7.1
Sports,Mid-Range,4064,6327,6277190.28,7.1


##### 3. Customer Segmentation Analysis (RFM)
RFM = Recency, Frequency, Monetary - a classic customer segmentation technique.

In [0]:
%sql
WITH rfm_base AS (
  SELECT
    customer_id,
    DATEDIFF(CURRENT_DATE(), MAX(order_date_key)) AS recency_days,
    COUNT(DISTINCT order_id) AS frequency,
    ROUND(SUM(line_total), 2) AS monetary
  FROM ecommerce_gold.fact_orders
  WHERE status NOT IN ('cancelled', 'returned')
  GROUP BY customer_id
),
rfm_scored AS (
  SELECT
    *,
    NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
    NTILE(5) OVER (ORDER BY frequency) AS f_score,
    NTILE(5) OVER (ORDER BY monetary) AS m_score
  FROM rfm_base
),
rfm_segments AS (
  SELECT
    *,
    CONCAT(r_score, f_score, m_score) AS rfm_cell,
    CASE
      WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
      WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal Customers'
      WHEN r_score >= 4 AND f_score <= 2 THEN 'New Customers'
      WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
      WHEN r_score <= 2 AND f_score <= 2 AND m_score >= 3 THEN 'Cant Lose Them'
      WHEN r_score <= 2 AND f_score <= 2 THEN 'Lost'
      ELSE 'Others'
    END AS segment
  FROM rfm_scored
)
SELECT
  segment,
  COUNT(*) AS customer_count,
  ROUND(AVG(monetary), 2) AS avg_spend,
  ROUND(AVG(frequency), 1) AS avg_orders,
  ROUND(AVG(recency_days), 0) AS avg_recency_days
FROM rfm_segments
GROUP BY segment
ORDER BY avg_spend DESC


segment,customer_count,avg_spend,avg_orders,avg_recency_days
Champions,1302,162285.5,6.3,141.0
Cant Lose Them,764,99347.82,2.3,297.0
At Risk,1674,88126.98,4.7,255.0
Loyal Customers,2904,69661.45,4.9,160.0
Others,698,45622.17,2.4,184.0
New Customers,976,44333.34,2.4,143.0
Lost,1482,10174.74,1.9,305.0


##### 4. Payment Method Analysis

In [0]:
%sql
SELECT
  payment_method,
  COUNT(DISTINCT order_id) AS total_orders,
  ROUND(SUM(line_total), 2) AS total_revenue,
  ROUND(AVG(line_total), 2) AS avg_order_value,
  ROUND(SUM(line_total) * 100.0 / SUM(SUM(line_total)) OVER (), 2) AS revenue_share_pct,
  ROUND(SUM(CASE WHEN status = 'cancelled' THEN line_total ELSE 0 END) * 100.0 /
        NULLIF(SUM(line_total), 0), 2) AS cancellation_rate_pct
FROM ecommerce_gold.fact_orders
GROUP BY payment_method
ORDER BY total_revenue DESC


payment_method,total_orders,total_revenue,avg_order_value,revenue_share_pct,cancellation_rate_pct
UPI,14284,2.6071628333E8,8745.35,27.98,10.92
Credit Card,7190,1.3722701082E8,9014.45,14.73,11.17
COD,7269,1.3528572317E8,8867.12,14.52,11.5
Net Banking,7145,1.3481810578E8,8960.4,14.47,11.36
Debit Card,7086,1.3284138674E8,8899.4,14.26,11.06
Wallet,7026,1.3092559173E8,8816.54,14.05,10.62


##### 5. Geographic Performance (Top Cities)

In [0]:
%sql
SELECT
  c.city,
  c.state,
  COUNT(DISTINCT c.customer_id) AS customers,
  COUNT(DISTINCT f.order_id) AS orders,
  ROUND(SUM(f.line_total), 2) AS revenue,
  ROUND(SUM(f.line_total) / COUNT(DISTINCT c.customer_id), 2) AS revenue_per_customer
FROM ecommerce_gold.fact_orders f
JOIN ecommerce_gold.dim_customers c ON f.customer_id = c.customer_id
WHERE f.status != 'cancelled'
GROUP BY c.city, c.state
ORDER BY revenue DESC
LIMIT 15


city,state,customers,orders,revenue,revenue_per_customer
Chennai,Tamil Nadu,717,3244,6.03395102E7,84155.52
Surat,Gujarat,701,3122,6.001907189E7,85619.22
Bhopal,Madhya Pradesh,641,2932,5.790708415E7,90338.66
Delhi,Delhi,650,2922,5.789370545E7,89067.24
Lucknow,Uttar Pradesh,675,3102,5.786041425E7,85719.13
Mumbai,Maharashtra,692,3164,5.737810616E7,82916.34
Kolkata,West Bengal,666,2962,5.569893322E7,83632.03
Bangalore,Karnataka,668,3043,5.516052743E7,82575.64
Kochi,Kerala,680,2962,5.428269186E7,79827.49
Ahmedabad,Gujarat,647,2898,5.402186968E7,83495.93


##### 6. Day-of-Week Ordering Patterns

In [0]:
%sql
SELECT
  d.day_name,
  d.day_of_week,
  COUNT(DISTINCT f.order_id) AS orders,
  ROUND(SUM(f.line_total), 2) AS revenue,
  ROUND(AVG(f.line_total), 2) AS avg_item_value,
  d.is_weekend
FROM ecommerce_gold.fact_orders f
JOIN ecommerce_gold.dim_date d ON f.order_date_key = d.date_key
WHERE f.status != 'cancelled'
GROUP BY d.day_name, d.day_of_week, d.is_weekend
ORDER BY d.day_of_week


day_name,day_of_week,orders,revenue,avg_item_value,is_weekend
Sunday,1,6375,1.2009515794E8,8879.49,true
Monday,2,6293,1.2273056702E8,9250.12,false
Tuesday,3,6377,1.1659933544E8,8691.07,false
Wednesday,4,6385,1.1904575653E8,9035.73,false
Thursday,5,6306,1.1552609781E8,8740.72,false
Friday,6,6312,1.1305916769E8,8514.13,false
Saturday,7,6293,1.2150635589E8,9207.82,true


##### 7. Top 10 Products by Revenue

In [0]:
%sql
SELECT
  p.product_name,
  p.category,
  p.price_tier,
  p.total_units_sold,
  p.total_revenue,
  p.revenue_rank,
  p.rating
FROM ecommerce_gold.dim_products p
WHERE p.revenue_rank <= 10
ORDER BY p.revenue_rank

product_name,category,price_tier,total_units_sold,total_revenue,revenue_rank,rating
Laptop,Electronics,Luxury,725,8.931146435000013E7,1,4.1
Laptop,Electronics,Luxury,744,7.026732944000009E7,2,3.1
Laptop,Electronics,Luxury,813,5.046565289000016E7,3,4.8
Laptop,Electronics,Luxury,785,4.654470993000005E7,4,3.9
Laptop,Electronics,Luxury,863,4.648295455999984E7,5,3.8
Smartphone,Electronics,Luxury,831,4.48941093300001E7,6,2.9
Tablet,Electronics,Luxury,803,4.2575772460000075E7,7,4.8
Smartphone,Electronics,Luxury,769,3.5066913659999944E7,8,4.2
Smartphone,Electronics,Luxury,714,2.9847648849999953E7,9,2.8
Tablet,Electronics,Luxury,797,2.9759878829999883E7,10,4.6


##### 8. Customer Lifetime Value Distribution

In [0]:
%sql
SELECT
  clv_tier,
  COUNT(*) AS customers,
  ROUND(MIN(total_revenue), 2) AS min_revenue,
  ROUND(AVG(total_revenue), 2) AS avg_revenue,
  ROUND(MAX(total_revenue), 2) AS max_revenue,
  ROUND(AVG(num_orders), 1) AS avg_orders,
  ROUND(AVG(unique_products), 1) AS avg_products_bought
FROM ecommerce_gold.agg_customer_clv
GROUP BY clv_tier
ORDER BY avg_revenue DESC


clv_tier,customers,min_revenue,avg_revenue,max_revenue,avg_orders,avg_products_bought
High,407,6059.23,225049.07,830717.75,3.5,8.3
Medium,3912,2558.87,141918.07,522779.26,5.1,11.0
Low,5575,102.81,32606.99,116731.69,4.1,8.0


##### 9. Month-over-Month Growth

In [0]:
%sql
WITH monthly AS (
  SELECT
    order_month,
    ROUND(SUM(net_revenue), 2) AS monthly_revenue,
    SUM(total_orders) AS monthly_orders
  FROM ecommerce_gold.agg_daily_revenue
  GROUP BY order_month
)
SELECT
  order_month,
  monthly_revenue,
  monthly_orders,
  LAG(monthly_revenue) OVER (ORDER BY order_month) AS prev_month_revenue,
  ROUND(
    (monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY order_month))
    * 100.0 / NULLIF(LAG(monthly_revenue) OVER (ORDER BY order_month), 0),
  2) AS revenue_growth_pct
FROM monthly
ORDER BY order_month


order_month,monthly_revenue,monthly_orders,prev_month_revenue,revenue_growth_pct
1,7.09411484E7,4280,null,null
2,6.461906424E7,3933,7.09411484E7,-8.91
3,6.963013316E7,4131,6.461906424E7,7.75
4,6.55195252E7,4140,6.963013316E7,-5.9
5,7.110173297E7,4299,6.55195252E7,8.52
6,6.774401548E7,4033,7.110173297E7,-4.72
7,7.238075842E7,4387,6.774401548E7,6.84
8,7.092516779E7,4239,7.238075842E7,-2.01
9,6.343552965E7,4028,7.092516779E7,-10.56
10,6.891056852E7,4169,6.343552965E7,8.63


##### 10. Product Category Cross-Sell Analysis

In [0]:
%sql
WITH order_categories AS (
  SELECT DISTINCT
    f.order_id,
    p.category
  FROM ecommerce_gold.fact_orders f
  JOIN ecommerce_gold.dim_products p ON f.product_id = p.product_id
  WHERE f.status NOT IN ('cancelled', 'returned')
)
SELECT
  a.category AS bought_category,
  b.category AS also_bought_category,
  COUNT(DISTINCT a.order_id) AS co_occurrence_count
FROM order_categories a
JOIN order_categories b ON a.order_id = b.order_id AND a.category < b.category
GROUP BY a.category, b.category
HAVING co_occurrence_count >= 50
ORDER BY co_occurrence_count DESC
LIMIT 20


bought_category,also_bought_category,co_occurrence_count
Books,Grocery,1921
Beauty,Grocery,1911
Beauty,Books,1908
Clothing,Grocery,1902
Books,Toys,1898
Clothing,Home & Kitchen,1878
Home & Kitchen,Toys,1876
Grocery,Toys,1873
Grocery,Sports,1860
Electronics,Home & Kitchen,1858


##### Pipeline Complete

This concludes the E-Commerce Data Lakehouse pipeline. Summary of what was built:

| Layer | Tables | Purpose |
|-------|--------|---------|
| **Landing** | 4 tables | Synthetic raw source data |
| **Bronze** | 4 tables | Raw ingestion with audit metadata |
| **Silver** | 4 tables | Cleansed, deduplicated, typed |
| **Gold** | 6 tables | Star schema + aggregates |
| **Audit** | 2 tables | DQ scores + quarantine |
| **Incremental** | 3 tables | CDC / MERGE patterns |

**Total: 23 Delta tables across 6 schemas**

In [0]:
schemas = [
    "ecommerce_landing",
    "ecommerce_bronze",
    "ecommerce_silver",
    "ecommerce_gold",
    "ecommerce_audit",
    "ecommerce_incremental",
]

print(f"{'Schema':<26}{'Tables':>8}")
print("-" * 34)

total = 0
for schema in schemas:
    try:
        count = spark.sql(f"SHOW TABLES IN {schema}").count()
        print(f"{schema:<26}{count:>8}")
        total += count
    except Exception:
        print(f"{schema:<26}{'(not found)':>8}")

print("-" * 34)
print(f"{'TOTAL':<26}{total:>8}")


Schema                      Tables
----------------------------------
ecommerce_landing                4
ecommerce_bronze                 4
ecommerce_silver                 4
ecommerce_gold                   6
ecommerce_audit                  1
ecommerce_incremental            3
----------------------------------
TOTAL                           22
